In [15]:
from notebooks.consts import PROCESSED_CSV
import pandas as pd

asoptimizer_updated_df = pd.read_csv(PROCESSED_CSV)

In [16]:
asoptimizer_updated_df.columns

Index(['Unnamed: 0', 'ISIS', 'Transfection', 'Linkage', 'Linkage_Location',
       'Canonical Gene Name', 'Cell_line', 'ASO_volume(nM)',
       'Treatment_Period(hours)', 'Sequence', 'Modification',
       'Chemical_Pattern', 'Inhibition(%)', 'probe_std', 'probe_count',
       'total_replicate_count', 'Cohort', 'index_v2', 'Cell line organism'],
      dtype='object')

In [17]:
# 1. Define target linkage types for filtering
target_linkages = ['phosphorothioate', 'phosphorothioate/phosphodiester']

# 2. Perform the filtering
initial_count = len(asoptimizer_updated_df)
asoptimizer_updated_df = asoptimizer_updated_df[asoptimizer_updated_df['Linkage'].isin(target_linkages)].copy()
final_count = len(asoptimizer_updated_df)
print(f"Total rows before filtering: {initial_count}")
print(f"Total rows after filtering:  {final_count}")

Total rows before filtering: 19078
Total rows after filtering:  19078


In [18]:
from notebooks.data.ASOptimizer.utility import transform_pattern_to_oligo, transform_linkage_to_oligo
from tauso.data.consts import *

asoptimizer_updated_df['split'] = 'test'
asoptimizer_updated_df['inhibition_percent'] = asoptimizer_updated_df[INHIBITION]
asoptimizer_updated_df['aso_sequence_5_to_3'] = asoptimizer_updated_df[SEQUENCE]
asoptimizer_updated_df['dosage'] = asoptimizer_updated_df[VOLUME]
asoptimizer_updated_df['sugar_mods'] = asoptimizer_updated_df[CHEMICAL_PATTERN].apply(transform_pattern_to_oligo)
asoptimizer_updated_df['backbone_mods'] = asoptimizer_updated_df.apply(
    lambda row: transform_linkage_to_oligo(
        row['Linkage_Location'],
        len(row[SEQUENCE])
    ),
    axis=1
)


In [19]:
from tauso.genome.read_human_genome import get_locus_to_data_dict
from notebooks.preprocessing import get_unique_genes

genes_u = get_unique_genes(asoptimizer_updated_df)
gene_to_data = get_locus_to_data_dict(include_introns=True, gene_subset=genes_u)

[Task] finished in 0.0025s
Elapsed DB:  0.0025441646575927734
Elapsed Fasta:  0.0025441646575927734
Length:  3267117988


In [20]:
import pandas as pd

# Translation table: converts A->U, C->G, G->C, T/U->A
COMP_U_TABLE = bytes.maketrans(b"ACGTUacgtu", b"UGCAaugcaa")

def get_antisense_u(seq: str) -> str:
    """Returns the reverse complement RNA (sense) sequence."""
    return seq.encode().translate(COMP_U_TABLE)[::-1].decode()

def find_sense_start(row, gene_to_data):
    """Finds the 0-indexed start position of the sense sequence in the mRNA."""
    gene = row[CANONICAL_GENE]
    antisense_seq = row[SEQUENCE]

    # Return -1 if gene isn't in our dictionary
    if pd.isna(gene) or gene not in gene_to_data:
        return -1

    # Get the mRNA and ensure it's formatted as uppercase RNA
    mrna = gene_to_data[gene].full_mrna.upper().replace('T', 'U')

    # Convert the antisense input into its sense counterpart
    sense_seq = get_antisense_u(antisense_seq)

    # .find() natively returns the starting index, or -1 if not found
    return mrna.find(sense_seq)

def get_populated_df_with_sense_start(df, gene_to_data):
    """Main wrapper to apply the logic to the DataFrame."""
    # Work on a copy to prevent SettingWithCopy warnings
    result_df = df.copy()

    result_df[SENSE_START] = result_df.apply(
        lambda row: find_sense_start(row, gene_to_data),
        axis=1
    )

    return result_df

In [21]:
asoptimizer_updated_df = get_populated_df_with_sense_start(asoptimizer_updated_df, gene_to_data)

In [22]:
len(asoptimizer_updated_df)

19078

In [23]:
asoptimizer_updated_df = asoptimizer_updated_df[asoptimizer_updated_df[SENSE_START] != -1]

In [24]:
len(asoptimizer_updated_df)

18153

In [25]:
import pandas as pd

def extract_rna_context(row, gene_to_data):
    """
    Extracts 50nt + sequence + 50nt from the mRNA using the SENSE_START index.
    """
    sense_start = row[SENSE_START]

    # 1. Skip if the sequence was never found (SENSE_START == -1)
    if pd.isna(sense_start) or sense_start == -1:
        raise ValueError("Not found!!")

    gene = row[CANONICAL_GENE]
    seq_len = len(str(row[SEQUENCE]))

    # 2. Skip if gene is missing from our dictionary
    if pd.isna(gene) or gene not in gene_to_data:
        raise ValueError("Not found!! gene missing")

    # 3. Get the mRNA and ensure it is formatted as uppercase RNA
    mrna = gene_to_data[gene].full_mrna.upper().replace('T', 'U')

    # 4. Calculate the strict boundaries
    # Using max(0, ...) prevents negative indexing if start is near the 5' end
    # Using min(len, ...) prevents out-of-bounds if near the 3' end
    left_bound = max(0, sense_start - 50)
    right_bound = min(len(mrna), sense_start + seq_len + 50)

    # 5. Slice and return
    return mrna[left_bound:right_bound]

# --- Applying it to your DataFrame ---

# Assuming your dataframe is named `all_data` from the previous step
asoptimizer_updated_df['rna_context'] = asoptimizer_updated_df.apply(
    lambda row: extract_rna_context(row, gene_to_data),
    axis=1
)

# Quick check to see the lengths (should be mostly seq_len + 100)
print(asoptimizer_updated_df[[SENSE_START, SEQUENCE, 'rna_context']].head())

   sense_start              Sequence  \
0         4447      GTCTAGTGTCATGGAA   
1          112  TCAGCACGCACACGGCCTTC   
2          112  TCAGCACGCACACGGCCTTC   
3          112  TCAGCACGCACACGGCCTTC   
4          112  TCAGCACGCACACGGCCTTC   

                                         rna_context  
0  CGCAGCAUUCACUAACUCGUAACUCUCCUUCCCUUCCUCCAAACAC...  
1  UGCAGUCCUCGGAACCAGGACCUCGGCGUGGCCUAGCGAGUUAUGG...  
2  UGCAGUCCUCGGAACCAGGACCUCGGCGUGGCCUAGCGAGUUAUGG...  
3  UGCAGUCCUCGGAACCAGGACCUCGGCGUGGCCUAGCGAGUUAUGG...  
4  UGCAGUCCUCGGAACCAGGACCUCGGCGUGGCCUAGCGAGUUAUGG...  


In [26]:
asoptimizer_updated_df['custom_id'] = asoptimizer_updated_df[CANONICAL_GENE].astype(str) + '_' + asoptimizer_updated_df[CELL_LINE].astype(str)

In [27]:
from notebooks.data.ASOptimizer.utility import transfection_to_oligo

asoptimizer_updated_df = transfection_to_oligo(asoptimizer_updated_df)

In [28]:
asoptimizer_updated_df.to_csv('asoptimizer_adapted.csv')